# Battery Air Cooling Simulator - Demo Notebook

This notebook demonstrates the main features of the battery air cooling simulator.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from battery_aircooling import SimulationConfig, ROMSolver, calculate_metrics
from battery_aircooling.physics_post import (
    summarize_results,
    compare_configurations,
    calculate_pareto_front
)
from battery_aircooling.visualize import (
    plot_temperature_distribution,
    plot_channel_flow_distribution,
    plot_sweep_results,
    plot_pareto_front,
    plot_spider_chart
)

%matplotlib inline

## 2. Single Simulation Example

In [ ]:
# Load configuration
config = SimulationConfig.from_yaml("study_rib_sweep.yaml")

print(f"Configuration: {config.name}")
print(f"Rib type: {config.ribs.type}")
print(f"Rib height: {config.ribs.height * 1000:.2f} mm")
print(f"Rib pitch: {config.ribs.pitch * 1000:.2f} mm")

In [ ]:
# Run simulation
solver = ROMSolver(config)
results = solver.solve()

print(f"Converged: {results.converged}")
print(f"Iterations: {results.iterations}")

In [ ]:
# Calculate metrics
Q_gen_total = config.cells.n_rows * config.cells.n_cols * config.cells.q_gen
metrics = calculate_metrics(results, config.metrics, Q_gen_total)

# Print summary
print(summarize_results(metrics))

In [ ]:
# Visualize temperature distribution
plot_temperature_distribution(results)

In [ ]:
# Visualize channel flow distribution
plot_channel_flow_distribution(results.channels)

## 3. Parameter Sweep: Rib Height

In [ ]:
# Define sweep parameters
rib_heights = np.linspace(0.001, 0.008, 10)  # 1mm to 8mm

print(f"Sweeping rib height: {len(rib_heights)} points")
print(f"Range: {rib_heights[0]*1000:.1f} - {rib_heights[-1]*1000:.1f} mm")

In [ ]:
# Run sweep
results_list = []
metrics_list = []

for height in rib_heights:
    # Create modified config
    config_copy = config.model_copy(deep=True)
    config_copy.ribs.height = height
    
    # Solve
    solver = ROMSolver(config_copy)
    result = solver.solve()
    
    # Calculate metrics
    m = calculate_metrics(result, config_copy.metrics, Q_gen_total)
    
    results_list.append(result)
    metrics_list.append(m)

print("Sweep completed!")

In [ ]:
# Plot sweep results
plot_sweep_results(
    rib_heights * 1000,  # Convert to mm
    metrics_list,
    "Rib Height",
    "mm"
)

## 4. Comparison: Different Rib Types

In [ ]:
# Compare rib types
rib_types = ["rect", "tri", "semi", "dimple"]
type_metrics = []

for rib_type in rib_types:
    config_copy = config.model_copy(deep=True)
    config_copy.ribs.type = rib_type
    config_copy.ribs.height = 0.004  # 4mm
    
    solver = ROMSolver(config_copy)
    result = solver.solve()
    
    m = calculate_metrics(result, config_copy.metrics, Q_gen_total)
    type_metrics.append(m)

print("Rib type comparison completed!")

In [ ]:
# Spider chart comparison
plot_spider_chart(type_metrics, rib_types)

In [ ]:
# Compare configurations
comparison = compare_configurations(type_metrics, rib_types)

print("\nBest Configurations:")
print(f"  Lowest Tmax: {comparison['best']['lowest_Tmax']}")
print(f"  Best Uniformity: {comparison['best']['best_uniformity']}")
print(f"  Lowest Power: {comparison['best']['lowest_power']}")
print(f"  Highest JF: {comparison['best']['highest_JF']}")
print(f"  Best Overall: {comparison['best']['best_overall']}")

## 5. Pareto Optimization

In [ ]:
# Find Pareto front (Tmax vs Ppump)
pareto_indices = calculate_pareto_front(metrics_list, "T_max", "P_pump")

print(f"Found {len(pareto_indices)} Pareto-optimal configurations")
print(f"Pareto indices: {pareto_indices}")

In [ ]:
# Plot Pareto front
config_names = [f"{h*1000:.1f}mm" for h in rib_heights]

plot_pareto_front(
    metrics_list,
    pareto_indices,
    config_names,
    objective1="T_max",
    objective2="P_pump"
)

## 6. Results Summary

In [ ]:
# Create summary table
import pandas as pd

summary_data = []
for height, metrics in zip(rib_heights, metrics_list):
    summary_data.append({
        'Height [mm]': height * 1000,
        'Tmax [°C]': metrics.T_max - 273.15,
        'dT [K]': metrics.dT_pack,
        'Uniformity': metrics.uniformity_index,
        'ΔP [Pa]': metrics.Delta_P,
        'Ppump [W]': metrics.P_pump,
        'JF [-]': metrics.JF_factor,
        'Objective': metrics.objective_score,
    })

df = pd.DataFrame(summary_data)
print("\nSummary Table:")
print(df.to_string(index=False))

In [ ]:
# Find optimal configuration
best_idx = df['Objective'].idxmin()
best_height = df.loc[best_idx, 'Height [mm]']

print(f"\n✨ Optimal Configuration:")
print(f"  Rib Height: {best_height:.2f} mm")
print(f"  Max Temperature: {df.loc[best_idx, 'Tmax [°C]']:.2f}°C")
print(f"  Pumping Power: {df.loc[best_idx, 'Ppump [W]']:.3f} W")
print(f"  JF Factor: {df.loc[best_idx, 'JF [-]']:.4f}")

## 7. Export Results

In [ ]:
# Save results to CSV
output_dir = Path("notebook_results")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "sweep_results.csv", index=False)
print(f"Results saved to: {output_dir / 'sweep_results.csv'}")

## Conclusions

This notebook demonstrated:

1. **Single simulation** - Running ROM solver and calculating metrics
2. **Parameter sweep** - Systematic exploration of rib height effects
3. **Rib type comparison** - Comparing different rib geometries
4. **Pareto optimization** - Finding optimal trade-offs
5. **Result visualization** - Various plots for analysis

### Key Findings:

- **Rib height** has significant impact on both heat transfer and pressure drop
- **Trade-off exists** between cooling performance and pumping power
- **Optimal configuration** depends on weighting of objectives
- **Pareto front** helps identify non-dominated designs